In [1]:
pip install langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os, json

In [3]:
# Initialize the splitter
# chunk_size is in characters (400 tokens * ~4 chars/token ≈ 1600 chars)
# chunk_overlap is the overlap between adjacent chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1600,
    chunk_overlap=200
)

OUTPUT_CHUNKS_DIR = "data/chunks"
os.makedirs(OUTPUT_CHUNKS_DIR, exist_ok=True)

def clean_text(text):
    """Remove known artifacts from scraped text."""
    # Remove click here artifacts
    text = text.replace("Click hereto download this medical illustration.", "")
    text = text.replace("Click here", "")
    # Remove written by sections
    if "WRITTEN BY" in text:
        text = text[:text.index("WRITTEN BY")]
    return text.strip()



In [4]:
def chunk_article(article):
    """Clean and chunk a single article into retrievable pieces."""

    # Clean first
    content = clean_text(article["content"])
    
    # Split into chunks
    chunks = splitter.split_text(content)
    
    
    # Each chunk gets the article metadata attached
    return [
        {   "chunk_id": f"{article['url']}::chunk_{i}",
            "text": chunk,
            "title": article["title"],
            "url": article["url"],
            "source": article["source"],
            "species": article["species"],
            "topic": article["topic"]
        }
        for i, chunk in enumerate(chunks)
    ]



In [5]:
# Process all articles
files = [f for f in os.listdir("data/raw") if f.endswith(".json")]
all_chunks = []

for f in files:
    with open(f"data/raw/{f}") as fp:
        article = json.load(fp)
    chunks = chunk_article(article)
    all_chunks.extend(chunks)

# Save all chunks to one file
with open(f"{OUTPUT_CHUNKS_DIR}/all_chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

print(f"Total articles: {len(files)}")
print(f"Total chunks: {len(all_chunks)}")
print(f"Average chunks per article: {len(all_chunks)//len(files)}")

Total articles: 330
Total chunks: 1462
Average chunks per article: 4


In [6]:
print(type(f))

<class '_io.TextIOWrapper'>


In [7]:
for f in files[:]: 
    with open(f"data/raw/{f}") as fp:
        article = json.load(fp)
    search_terms = ["conditions", "symptoms", "care"]
    if "species" in article and article["species"] == "cat" \
    and "topic" in article and article["topic"] in search_terms \
        and "shedding" in article["content"].lower():
        print(f"Title: {article['title']}")
        print(f"Topic: {article['topic']}")
        print(f"Text: {article['content'][:200]}...")  # Print first 200 chars of the chunk
        print("-" * 80)
